In [1]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import time

# ─────────────────────────────────────────────
# 1. Load 4 Word2Vec Models
# ─────────────────────────────────────────────
models = {}
print("[INFO] Loading 4 Word2Vec models...")

for i in range(1, 5):
    path = f"w2v_model_output_{i}.model"
    print(f"[INFO] Loading model {i}: {path}")
    models[i] = Word2Vec.load(path)

print("[INFO] All models loaded.")

# ─────────────────────────────────────────────
# 2. Load inference CSV
# ─────────────────────────────────────────────
base_path = "../../../../"
goto_folder = "ResultGroup/3.Circuit/"
filename = "Circuit-QuantumVLM-2.5VL-32B-v2.1060.csv"

print("[INFO] Loading inference CSV...")
df = pd.read_csv(f"{base_path}{goto_folder}{filename}")
print("[INFO] Rows:", len(df))

# Columns for predictions and outputs
pred_cols = [f"prediction_{i}" for i in range(1, 5)]
gt_cols   = [f"output_{i}"     for i in range(1, 5)]
score_cols = [f"w2v_score_{i}" for i in range(1, 5)]

# ─────────────────────────────────────────────
# 3. Convert sentence to vector (model-specific)
# ─────────────────────────────────────────────
def sentence_vector(text, model):
    words = str(text).lower().split()
    vecs = [model.wv[w] for w in words if w in model.wv]

    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# ─────────────────────────────────────────────
# 4. Compute Word2Vec scores for 4 pairs
# ─────────────────────────────────────────────
print("[INFO] Scoring...")
total = len(df)
last_print = time.time()

for idx, row in df.iterrows():

    for i, (pcol, gcol, scol) in enumerate(zip(pred_cols, gt_cols, score_cols), start=1):

        model = models[i]  # choose correct model
        v1 = sentence_vector(row[pcol], model)
        v2 = sentence_vector(row[gcol], model)

        sim = cosine_similarity([v1], [v2])[0][0]
        df.at[idx, scol] = sim

    if time.time() - last_print > 1:
        pct = (idx+1) / total * 100
        print(f"[INFO] {idx+1}/{total} ({pct:.2f}%)")
        last_print = time.time()

print("[INFO] Scoring complete.")

# ─────────────────────────────────────────────
# 5. Save results
# ─────────────────────────────────────────────
out_path = "output/3-4-2-3-2.word2vec-circuit-quantumvlm-25vl-v2.1060.csv"
df.to_csv(out_path, index=False)

print("[INFO] Saved to:", out_path)

# ─────────────────────────────────────────────
# 6. Print mean scores
# ─────────────────────────────────────────────
print("\n=== Mean W2V Scores (per model) ===")
for scol in score_cols:
    print(f"{scol}: {df[scol].mean():.4f}")

overall_mean = df[score_cols].mean(axis=1).mean()
print("\nOverall Mean (all 4 models combined):", overall_mean)


[INFO] Loading 4 Word2Vec models...
[INFO] Loading model 1: w2v_model_output_1.model
[INFO] Loading model 2: w2v_model_output_2.model
[INFO] Loading model 3: w2v_model_output_3.model
[INFO] Loading model 4: w2v_model_output_4.model
[INFO] All models loaded.
[INFO] Loading inference CSV...
[INFO] Rows: 941
[INFO] Scoring...
[INFO] 151/941 (16.05%)
[INFO] 284/941 (30.18%)
[INFO] 415/941 (44.10%)
[INFO] 547/941 (58.13%)
[INFO] 682/941 (72.48%)
[INFO] 816/941 (86.72%)
[INFO] Scoring complete.
[INFO] Saved to: output/3-4-2-3-2.word2vec-circuit-quantumvlm-25vl-v2.1060.csv

=== Mean W2V Scores (per model) ===
w2v_score_1: 0.9321
w2v_score_2: 0.9244
w2v_score_3: 0.9076
w2v_score_4: 0.8410

Overall Mean (all 4 models combined): 0.901265797522164
